# 04 · Statistical Testing & Predictive Benchmarks

> **Stage 5a + Path B (STRUCTURE.md).** Two goals: **(1)** quantify group differences with *effect sizes*
> (with 2M rows, p-values are meaninglessly small — effect size is the finding); **(2)** run the predictive
> tasks from assessment §6 and honestly separate the **legitimate benchmark** (attrition) from the
> **cleaning artifacts** (salary, performance).

> **Governing caveat:** a model looking strong here is a red flag, not a win. Salary was median-filled from
> Level×Dept, so predicting it recovers the fill — R² near 1 is mechanical, not learned.

In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import mck_style
mck_style.apply()
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

ARCH = "../archive"; PROC = "../data/processed"; FIG = "../reports/figures"
os.makedirs(PROC, exist_ok=True); os.makedirs(FIG, exist_ok=True)
np.random.seed(42)
print("Setup complete · pandas", pd.__version__)

Setup complete · pandas 3.0.3


In [2]:
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, accuracy_score, confusion_matrix

df = pd.read_parquet(f"{PROC}/hr_features.parquet")
# A reproducible 200k analytic sample keeps sklearn fast without changing conclusions
samp = df.sample(200_000, random_state=42).reset_index(drop=True)
print("Full:", df.shape, "| Analytic sample:", samp.shape)

Full: (2000000, 26) | Analytic sample: (200000, 26)


## 5a · Statistical testing — effect size over p-value

**Test 1 — Does salary differ by Job_Level?** Kruskal–Wallis (non-parametric; robust at scale) + ε² effect size.

In [3]:
groups = [g["Salary"].values for _,g in df.groupby("Job_Level")]
H, p = stats.kruskal(*groups)
k, n = df["Job_Level"].nunique(), len(df)
eps2 = (H - k + 1) / (n - k)   # epsilon-squared effect size
print(f"Kruskal-Wallis H={H:,.0f}  p={p:.2e}  epsilon^2={eps2:.3f}  (>=0.14 = large)")

Kruskal-Wallis H=1,756,448  p=0.00e+00  epsilon^2=0.878  (>=0.14 = large)


**Test 2 — Is attrition (`Status`) associated with `Department`?** Chi-square of independence + Cramér's V.

In [4]:
ct = pd.crosstab(df["Department"], df["Status"])
chi2, pchi, dof, _ = stats.chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (len(df) * (min(ct.shape)-1)))
print(f"Chi-square={chi2:,.1f}  p={pchi:.2e}  Cramer's V={cramers_v:.4f}  (<0.1 = negligible)")

Chi-square=1,915.0  p=0.00e+00  Cramer's V=0.0179  (<0.1 = negligible)


> **So What:** salary vs. level is a **large** effect (ε² well above 0.14) — the pay ladder is real *within
> the dataset's construction*. But Status vs. Department is a **negligible** association (Cramér's V ≈ 0), even
> though p ≈ 0. **Implication:** the giant sample makes everything "significant"; only effect size tells the
> truth — and it says attrition carries no departmental signal.

## Path B, Model 1 · Attrition classifier — the legitimate benchmark

Target: `Attrition_Flag` (Resigned/Terminated = 1 vs. Active = 0; the 453 Retired rows dropped). Features are
strictly non-leaky: `Age, Experience_Years, Salary, Department, Job_Level, Work_Mode`. We compare a
majority-class baseline against logistic regression.

In [5]:
m = samp[samp["Status"] != "Retired"].copy()
num_f = ["Age","Experience_Years","Salary"]
cat_f = ["Department","Job_Level","Work_Mode"]
X, y = m[num_f+cat_f], m["Attrition_Flag"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

pre = ColumnTransformer([("num", StandardScaler(), num_f),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), cat_f)])
clf = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000))]).fit(Xtr, ytr)
base = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)

p_lr = clf.predict_proba(Xte)[:,1]
res = pd.DataFrame({
    "model": ["Baseline (majority)","Logistic regression"],
    "ROC-AUC": [0.5, roc_auc_score(yte, p_lr)],
    "PR-AUC":  [yte.mean(), average_precision_score(yte, p_lr)],
    "Accuracy":[accuracy_score(yte, base.predict(Xte)), accuracy_score(yte, clf.predict(Xte))],
}).round(4)
res

,model,ROC-AUC,PR-AUC,Accuracy
0,Baseline (majority),0.50,0.10,0.90
1,Logistic regression,0.67,0.17,0.90


In [6]:
print("Attrition rate by seniority — the source of the signal (%):")
print((m.groupby("Job_Level")["Attrition_Flag"].mean()*100).round(1)
        .reindex(["Junior","Mid","Senior","Director"]).to_string())

Attrition rate by seniority — the source of the signal (%):
Job_Level
Junior     16.60
Mid         7.70
Senior      4.00
Director    5.50


> **So What:** logistic regression lifts **ROC-AUC to ≈0.67**, clearly beating the 0.50 baseline — there *is*
> a modest, real attrition signal. It comes from **seniority/tenure** (Junior staff churn ~16.6% vs ~4% at
> Senior level; low-experience staff leave far more), **not** department (Cramér's V ≈ 0.02 above).
> **Implication:** this is the legitimate model of the three — report it as *"modest, seniority-driven signal"*
> and target retention at early-career staff. Accuracy stays at the 90% base rate only because the classes are
> imbalanced; AUC/PR-AUC are the honest metrics here.

## Path B, Model 2 · Salary model — the cleaning artifact (⚠️ not a win)

We predict `Salary` from `Job_Level + Department` alone. Because salaries were **median-filled by exactly these
two keys**, the model reconstructs the fill.

In [7]:
Xs = samp[["Job_Level","Department"]]; ys = samp["Salary"]
Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.25, random_state=42)
sal = Pipeline([("oh", OneHotEncoder(handle_unknown="ignore")), ("lr", LinearRegression())]).fit(Xtr, ytr)
r2 = r2_score(yte, sal.predict(Xte))
print(f"Salary ~ Job_Level + Department   ->   R^2 = {r2:.3f}")
print("Interpretation: R^2 this high from two categoricals = recovering the median-fill, NOT real signal.")

Salary ~ Job_Level + Department   ->   R^2 = 0.949
Interpretation: R^2 this high from two categoricals = recovering the median-fill, NOT real signal.


> **So What:** two categorical columns explain almost all salary variance — impossible with genuine market
> pay, inevitable with median-filled pay. **Implication:** a portfolio writeup must **label this an artifact**;
> presenting it as a "salary prediction model (R²≈0.9+)" would be misleading.

## Path B, Model 3 · Performance rating — confirming there is no signal

In [8]:
from sklearn.ensemble import RandomForestClassifier
mp = samp.copy()
Xp = mp[num_f+["Department","Job_Level","Work_Mode"]]; yp = mp["Performance_Rating"].astype(str)
Xtr, Xte, ytr, yte = train_test_split(Xp, yp, test_size=0.25, stratify=yp, random_state=42)
rf = Pipeline([("pre", pre), ("rf", RandomForestClassifier(n_estimators=60, max_depth=8,
              n_jobs=-1, random_state=42))]).fit(Xtr, ytr)
acc = accuracy_score(yte, rf.predict(Xte))
majority = yte.value_counts(normalize=True).max()
print(f"Random-forest accuracy: {acc:.4f}  |  majority-class baseline: {majority:.4f}")
print("Model does not beat 'always predict Good' -> ratings carry no learnable structure.")

Random-forest accuracy: 0.5003  |  majority-class baseline: 0.5003
Model does not beat 'always predict Good' -> ratings carry no learnable structure.


> **So What:** the classifier cannot beat *always guess "Good"* — ratings were stratified-sampled, so no
> attribute predicts them. **Implication:** correctly detecting *absence* of signal is the deliverable here;
> it protects downstream users from over-trusting a performance model built on this file.

### Stage 5a / Path B — Gate checklist ✅
- [x] Effect sizes (ε², Cramér's V) reported alongside p-values; sample-size inflation called out
- [x] Attrition benchmark beats baseline — modest ROC-AUC ≈ 0.67, seniority-driven (no leakage in features)
- [x] Salary R²≈1 explicitly **labelled a median-fill artifact**, not a modelling win
- [x] Performance model shown to carry no signal (does not beat majority baseline)
- [x] `random_state=42` throughout; 200k analytic sample documented

→ Proceed to **05 · Reporting**.